# Global CO2 Emissions: Exploratory Data Analysis (EDA) Report

---

## Project Overview

| Item | Description |
|------|-------------|
| **Objective** | Analyze global CO2 emission trends and identify key patterns |
| **Dataset** | World Bank / Our World in Data (1960-2023) |
| **Data Source** | [data360.worldbank.org](https://data360.worldbank.org/en/dataset/OWID_CB) |
| **Live Dashboard** | [global-co2-insight-by-ju-ho-kim.streamlit.app](https://global-co2-insight-by-ju-ho-kim.streamlit.app/) |
| **Author** | Ju Ho Kim |

---

## Executive Summary

This analysis explores **64 years of global CO2 emissions data** across 205 countries to answer two key questions:

1. **Who are the largest emitters?** → Volume Analysis
2. **Who is growing the fastest?** → Growth Analysis

### Key Findings (Preview)
- **China** dominates global emissions (~12M kt in 2023), surpassing the US around 2006
- **Southeast Asian nations** (Lao PDR, Cambodia, Vietnam) show explosive growth rates (>1000%)
- **Developed nations** (UK, Germany) show declining emissions over the past decade

---

## Table of Contents

1. [Setup / Libraries](#1-setup--libraries)
2. [Data Loading / Inspection](#2-data-loading--inspection)
3. [Data Cleaning](#3-data-cleaning)
4. [Volume Analysis: Top Emitters](#4-volume-analysis-top-emitters)
5. [Growth Analysis: Fastest Growing](#5-growth-analysis-fastest-growing)
6. [Comparative Analysis](#6-comparative-analysis)
7. [Key Findings / Conclusions](#7-key-findings--conclusions)
8. [Limitations / Future Work](#8-limitations--future-work)

---

## 1. Setup / Libraries

I originally used **Plotly** for interactivity, but here I used **Matplotlib** and **Seaborn** to generate static charts to ensure they render properly on GitHub.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Color palettes
COLORS_VOLUME = ['#E63946', '#F4A261', '#2A9D8F', '#264653', '#E9C46A']
COLORS_GROWTH = ['#7209B7', '#3A0CA3', '#4361EE', '#4CC9F0', '#72EFDD']

print("Setup complete!")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")

---

## 2. Data Loading / Inspection

The dataset contains annual CO2 emissions (in kilotonnes) for countries worldwide from 1960 to 2023.

In [ ]:
# Load dataset
df = pd.read_csv('../data/co2_emissions_kt_by_country_2023.csv')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Year Range: {df['year'].min()} - {df['year'].max()}")
print(f"Unique Countries/Regions: {df['country_name'].nunique()}")
print(f"\nColumns: {df.columns.tolist()}")
print("="*60)

In [ ]:
# Preview data
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Data types and missing values
print("\nData Types & Missing Values:")
print("-"*40)
for col in df.columns:
    missing = df[col].isna().sum()
    pct = missing / len(df) * 100
    print(f"{col:15} | {str(df[col].dtype):10} | Missing: {missing:,} ({pct:.1f}%)")

In [ ]:
# Basic statistics
print("\nEmission Value Statistics (kt):")
df['value'].describe().apply(lambda x: f"{x:,.2f}")

---

## 3. Data Cleaning

### Problem Statement

The raw dataset contains **aggregate regions** (e.g., "World", "OECD Members", "High Income") mixed with actual countries. These must be removed to avoid double-counting in our analysis.

### Cleaning Strategy

Applied a **three-layer filtering system**:

| Layer | Method | Examples Removed |
|-------|--------|------------------|
| 1 | Explicit name list | "World", "European Union", "OECD Members" |
| 2 | Country code list | "WLD", "EUU", "OED" |
| 3 | Pattern matching | Names containing "income", "IBRD", "excluding" |

> **Note:** This cleaning logic is **synced with `src/app.py`** to ensure consistency between the EDA and the live dashboard.

In [ ]:
# ============================================
# ROBUST CLEANING LOGIC (Synced with app.py)
# ============================================

# Layer 1: Explicit name list
ignore_list = [
    'World', 'Arab World', 'Central Europe and the Baltics',
    'East Asia & Pacific', 'East Asia & Pacific (excluding high income)', 
    'East Asia & Pacific (IDA & IBRD countries)',
    'Europe & Central Asia', 'Europe & Central Asia (excluding high income)', 
    'Europe & Central Asia (IDA & IBRD countries)',
    'European Union', 'Euro area',
    'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)', 
    'Latin America & the Caribbean (IDA & IBRD countries)',
    'Middle East & North Africa', 'Middle East & North Africa (excluding high income)', 
    'Middle East & North Africa (IDA & IBRD countries)',
    'North America', 'South Asia', 'South Asia (IDA & IBRD)',
    'Sub-Saharan Africa', 'Sub-Saharan Africa (excluding high income)', 
    'Sub-Saharan Africa (IDA & IBRD countries)',
    'Africa Eastern and Southern', 'Africa Western and Central',
    'High income', 'Low & middle income', 'Low income', 'Lower middle income', 
    'Middle income', 'Upper middle income', 'OECD members',
    'IDA & IBRD total', 'IBRD only', 'IDA total', 'IDA blend', 'IDA only',
    'Heavily indebted poor countries (HIPC)', 
    'Least developed countries: UN classification',
    'Fragile and conflict affected situations', 
    'Early-demographic dividend', 'Late-demographic dividend', 
    'Pre-demographic dividend', 'Post-demographic dividend',
    'Small states', 'Pacific island small states', 'Caribbean small states', 
    'Other small states', 'Not classified',
    'Africa', 'Asia', 'Asia (excl. China and India)', 'Europe',
    'Europe (excl. EU-27)', 'Europe (excl. EU-28)', 
    'European Union (27)', 'European Union (28)',
    'North America (excl. USA)', 'Oceania', 'South America',
    'Non-OECD (GCP)', 'OECD (GCP)',
    'High-income countries', 'Low-income countries',
    'Lower-middle-income countries', 'Upper-middle-income countries'
]

# Layer 2: Country code list
aggregate_codes = [
    'ARB', 'CSS', 'EAP', 'EAS', 'ECA', 'ECS', 'EMU', 'EUU',
    'FCS', 'HIC', 'HPC', 'IBD', 'IBT', 'IDA', 'IDX', 'INX',
    'LAC', 'LCN', 'LDC', 'LIC', 'LMC', 'LMY', 'LTE', 'MEA',
    'MIC', 'MNA', 'NAC', 'OED', 'OSS', 'PRE', 'PSS', 'PST',
    'SAS', 'SSA', 'SSF', 'SST', 'TEA', 'TEC', 'TLA', 'TMN',
    'TSA', 'TSS', 'UMC', 'WLD',
    'OWID_AFR', 'OWID_ASI', 'OWID_EUR', 'OWID_EUN', 'OWID_NAM',
    'OWID_OCE', 'OWID_SAM', 'OWID_WRL', 'OWID_HIC', 'OWID_LIC',
    'OWID_LMC', 'OWID_UMC', 'OWID_NON_OECD', 'OWID_OECD'
]

# Layer 3: Pattern matching function
def looks_like_aggregate(name):
    """Returns True if the name appears to be an aggregate region."""
    if pd.isna(name):
        return True
    patterns = ['income', 'dividend', 'IBRD', 'IDA', 'excluding', 'excl.', 
               '(GCP)', 'European Union', 'OECD']
    return any(p in str(name) for p in patterns)

# Apply all filters
df_clean = df[
    (~df['country_name'].isin(ignore_list)) &
    (~df['country_code'].isin(aggregate_codes)) &
    (~df['country_name'].apply(looks_like_aggregate))
].copy()

# Report results
removed = len(df) - len(df_clean)
print("="*60)
print("DATA CLEANING RESULTS")
print("="*60)
print(f"Before cleaning: {len(df):,} rows")
print(f"After cleaning:  {len(df_clean):,} rows")
print(f"Removed:         {removed:,} rows ({removed/len(df)*100:.1f}%)")
print(f"\nCountries remaining: {df_clean['country_name'].nunique()}")
print("="*60)

In [ ]:
# Verify: Show top 10 emitters in latest year
latest_year = df_clean['year'].max()
top_10_verify = df_clean[df_clean['year'] == latest_year].nlargest(10, 'value')

print(f"\nVerification: Top 10 Emitters in {latest_year}")
print("-"*50)
for i, row in top_10_verify.iterrows():
    print(f"   {row['country_name']:25} {row['value']:>12,.0f} kt")

---

## 4. Volume Analysis: Top Emitters

### Research Question
> **Which countries emit the most CO2, and how have their emissions changed over time?**

I identified the **Top 5 emitters** based on the most recent year (2023) and analyze their historical trends.

In [ ]:
# Identify Top 5 emitters by volume
latest_year = df_clean['year'].max()
df_latest = df_clean[df_clean['year'] == latest_year]

top_5_volume = df_latest.nlargest(5, 'value')['country_name'].tolist()

print(f"Top 5 CO2 Emitters ({latest_year}):")
print("-"*50)
for i, country in enumerate(top_5_volume, 1):
    val = df_latest[df_latest['country_name'] == country]['value'].values[0]
    print(f"   {i}. {country:25} {val:>12,.0f} kt")

In [ ]:
# Chart 1: Historical Trends of Top 5 Emitters
fig, ax = plt.subplots(figsize=(14, 7))

for i, country in enumerate(top_5_volume):
    data = df_clean[df_clean['country_name'] == country]
    ax.plot(data['year'], data['value']/1e6, 
            label=country, linewidth=2.5, color=COLORS_VOLUME[i])

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('CO2 Emissions (Million kt)', fontsize=12)
ax.set_title(f'CO2 Emissions Trend: Top 5 Emitters (1960-{latest_year})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(1960, latest_year)
ax.set_ylim(0, None)

# Add annotation for China surpassing US
ax.axvline(x=2006, color='gray', linestyle='--', alpha=0.7)
ax.annotate('China surpasses US\n(~2006)', xy=(2006, 7), fontsize=9, 
            ha='center', color='gray')

plt.tight_layout()
plt.savefig('../assets/chart1_volume_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart 1 saved to: assets/chart1_volume_trend.png")

### Insight: Volume Analysis

- **China's dramatic rise:** China's emissions were relatively low until ~2000, then grew exponentially, surpassing the US around 2006.
- **US plateau:** The United States' emissions have plateaued since ~2005 and show a slight decline.
- **India's growth:** India has emerged as the 3rd largest emitter, with steady growth particularly after 2000.

In [ ]:
# Chart 2: Bar Chart - Top 10 Emitters (Latest Year)
top_10 = df_latest.nlargest(10, 'value')

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_10)))
bars = ax.barh(top_10['country_name'], top_10['value']/1e6, color=colors[::-1])

ax.set_xlabel('CO2 Emissions (Million kt)', fontsize=12)
ax.set_title(f'Top 10 CO2 Emitters ({latest_year})', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, top_10['value']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
            f'{val/1e6:.1f}M', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/chart2_top10_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart 2 saved to: assets/chart2_top10_bar.png")

---

## 5. Growth Analysis: Fastest Growing

### Research Question
> **Which countries have the highest CO2 emission growth rates, and what does this mean for future trends?**

### Methodology

Calculated the **10-year growth rate** using the formula:

$$\text{Growth Rate (\%)} = \frac{\text{Emissions}_{2023} - \text{Emissions}_{2013}}{\text{Emissions}_{2013}} \times 100$$

**Filter:** Only countries with emissions > 10,000 kt in 2023 are included to avoid misleading percentages from tiny base values (e.g., 10 → 20 kt = 100% growth).

In [ ]:
# Calculate growth rates
start_year = latest_year - 10
print(f"Analyzing growth: {start_year} → {latest_year}")

# Get emissions for start and end years
df_start = df_clean[df_clean['year'] == start_year].set_index('country_name')['value']
df_end = df_clean[df_clean['year'] == latest_year].set_index('country_name')['value']

# Filter: Only significant emitters (> 10,000 kt in latest year)
significant = df_end[df_end > 10000].index
valid_countries = df_start.index.intersection(significant)

# Calculate growth rate
growth_rate = ((df_end.loc[valid_countries] - df_start.loc[valid_countries]) 
               / df_start.loc[valid_countries] * 100).dropna()

# Top 5 fastest growing
top_5_growth = growth_rate.nlargest(5)
top_5_growth_list = top_5_growth.index.tolist()

# Bottom 5 (biggest declines)
bottom_5_growth = growth_rate.nsmallest(5)

print(f"\nTop 5 FASTEST Growing ({start_year}-{latest_year}):")
print("-"*55)
for i, (country, rate) in enumerate(top_5_growth.items(), 1):
    s_val = df_start.get(country, 0)
    e_val = df_end.get(country, 0)
    print(f"   {i}. {country:20} +{rate:>8,.1f}%  ({s_val:>10,.0f} → {e_val:>10,.0f} kt)")

print(f"\nTop 5 BIGGEST Declines ({start_year}-{latest_year}):")
print("-"*55)
for i, (country, rate) in enumerate(bottom_5_growth.items(), 1):
    s_val = df_start.get(country, 0)
    e_val = df_end.get(country, 0)
    print(f"   {i}. {country:20} {rate:>9,.1f}%  ({s_val:>10,.0f} → {e_val:>10,.0f} kt)")

In [ ]:
# Chart 3: Growth Trend - Top 5 Fastest Growing
fig, ax = plt.subplots(figsize=(14, 7))

for i, country in enumerate(top_5_growth_list):
    data = df_clean[(df_clean['country_name'] == country) & 
                    (df_clean['year'] >= start_year)]
    ax.plot(data['year'], data['value']/1e3, 
            label=f"{country} (+{top_5_growth[country]:.0f}%)",
            linewidth=2.5, marker='o', markersize=4, color=COLORS_GROWTH[i])

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('CO2 Emissions (Thousand kt)', fontsize=12)
ax.set_title(f'Fastest Growing Emitters ({start_year}-{latest_year})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(start_year, latest_year)

plt.tight_layout()
plt.savefig('../assets/chart3_growth_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart 3 saved to: assets/chart3_growth_trend.png")

### Insight: Growth Analysis

- **Developing nations lead growth:** Countries like Lao PDR, Cambodia, and Vietnam show explosive growth (>1000%), driven by rapid industrialization.
- **Base effect:** While growth *rates* are high, the absolute emissions are still relatively small compared to major emitters.
- **Policy implications:** These high-growth countries may become significant emitters in the coming decades without intervention.

---

## 6. Comparative Analysis

### Volume vs Growth: Different Stories

| Metric | Leaders | Insight |
|--------|---------|----------|
| **Volume** | China, USA, India | Established industrial economies |
| **Growth** | Lao PDR, Cambodia, Vietnam | Rapidly industrializing nations |

In [ ]:
# Chart 4: Side-by-Side Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Volume Leaders
ax1 = axes[0]
for i, country in enumerate(top_5_volume):
    data = df_clean[df_clean['country_name'] == country]
    ax1.plot(data['year'], data['value']/1e6, 
             label=country, linewidth=2.5, color=COLORS_VOLUME[i])
ax1.set_xlabel('Year', fontsize=11)
ax1.set_ylabel('CO2 Emissions (Million kt)', fontsize=11)
ax1.set_title(f'Top 5 by Volume ({latest_year})', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim(1960, latest_year)

# Right: Growth Leaders
ax2 = axes[1]
for i, country in enumerate(top_5_growth_list):
    data = df_clean[(df_clean['country_name'] == country) & 
                    (df_clean['year'] >= start_year)]
    ax2.plot(data['year'], data['value']/1e3, 
             label=f"{country}", linewidth=2.5, 
             marker='o', markersize=4, color=COLORS_GROWTH[i])
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('CO2 Emissions (Thousand kt)', fontsize=11)
ax2.set_title(f'Top 5 by Growth ({start_year}-{latest_year})', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim(start_year, latest_year)

plt.tight_layout()
plt.savefig('../assets/chart4_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart 4 saved to: assets/chart4_comparison.png")

---

## 7. Key Findings / Conclusions

### Summary Statistics

In [ ]:
# Final summary statistics
total_2023 = df_clean[df_clean['year'] == latest_year]['value'].sum()
china_2023 = df_clean[(df_clean['year'] == latest_year) & 
                       (df_clean['country_name'] == 'China')]['value'].values[0]
china_share = china_2023 / total_2023 * 100

print("="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total Global Emissions ({latest_year}): {total_2023/1e6:,.2f} Million kt")
print(f"China's Share: {china_share:.1f}%")
print(f"Countries Analyzed: {df_clean['country_name'].nunique()}")
print(f"Time Period: 1960 - {latest_year} ({latest_year - 1960 + 1} years)")
print("="*60)

### Key Takeaways

1. **Concentration of Emissions:** The top 5 emitters (China, USA, India, Russia, Japan) account for over 60% of global CO2 emissions.

2. **China's Dominance:** China alone represents ~32% of global emissions, having surpassed the US around 2006.

3. **Diverging Trends:**
   - Developed nations (US, EU) show stabilizing or declining emissions
   - Developing nations (Southeast Asia, Africa) show rapid growth

4. **Future Outlook:** If current growth rates continue, countries like Vietnam and Indonesia could become major emitters within the next two decades.

---

## 8. Limitations and Future Work

### Limitations

| Limitation | Impact |
|------------|--------|
| No per-capita analysis | Large population countries appear worse |
| No sectoral breakdown | Can't identify emission sources |
| Historical data quality | Early years may have measurement issues |

### Future Work

- **Per-capita analysis:** Incorporate population data for fairer comparison
- **Sectoral analysis:** Break down by industry, transport, residential
- **Predictive modeling:** Forecast future emissions using time series models
- **Policy correlation:** Analyze impact of climate policies on emissions

---

## Resources

- **Live Dashboard:** [global-co2-insight-by-ju-ho-kim.streamlit.app](https://global-co2-insight-by-ju-ho-kim.streamlit.app/)
- **Data Source:** [World Bank Data360](https://data360.worldbank.org/en/dataset/OWID_CB)
- **GitHub Repository:** [github.com/jurinho17-sv/global-co2-insight](https://github.com/jurinho17-sv/global-co2-insight)

---

*By Ju Ho Kim | Contact: juho_kim@berkeley.edu*